In [ ]:
# =============================================================================
# ENSEMBLE: Combine Multiple Models for Better Generalization
# Expected: 0.81-0.82 (beating all individual models!)
# Runtime: ~5 minutes        first ensemble tries 29/11 17:02
# =============================================================================

import numpy as np
import pandas as pd
import pickle
from datetime import datetime
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score

print("="*80)
print("ENSEMBLE: COMBINING BEST MODELS")
print("="*80)

ENSEMBLE: COMBINING BEST MODELS


In [2]:

# =============================================================================
# 1. LOAD DATA
# =============================================================================

print("\n--- Loading data ---")
X = pd.read_pickle('../../data/processed/X_train_processed.pkl')
y = pd.read_pickle('../../data/processed/y_train.pkl')
X_test = pd.read_pickle('../../data/processed/X_test_processed.pkl')
test_ids = pd.read_pickle('../../data/processed/test_ids.pkl')

print(f"X_train: {X.shape}")
print(f"y_train: {y.shape}")
print(f"X_test: {X_test.shape}")


--- Loading data ---
X_train: (20885, 95)
y_train: (20885,)
X_test: (5221, 95)


In [3]:

# =============================================================================
# 2. LOAD YOUR BEST MODELS
# =============================================================================

print("\n" + "="*80)
print("LOADING TRAINED MODELS")
print("="*80)

# Update these paths to match your saved models
models = {}

# Model 1: GB Optuna (0.801)
try:
    with open('../../outputs/models/gb_optuna_20251129_1645.pkl', 'rb') as f:
        models['gb_optuna'] = pickle.load(f)
    print("✓ Loaded: GB Optuna (Kaggle: 0.801)")
except:
    print("⚠️ GB Optuna not found - skipping")

# Model 2: XGBoost Optuna (0.795)
try:
    with open('../../outputs/models/xgboost_optuna_conservative_20251129_1409.pkl', 'rb') as f:
        models['xgb_optuna'] = pickle.load(f)
    print("✓ Loaded: XGBoost Optuna (Kaggle: 0.795)")
except:
    print("⚠️ XGBoost Optuna not found - skipping")

# Model 3: XGBoost Baseline (0.789)
# If you saved it, load it here
# try:
#     with open('path_to_xgb_baseline.pkl', 'rb') as f:
#         models['xgb_baseline'] = pickle.load(f)
#     print("✓ Loaded: XGBoost Baseline")
# except:
#     print("⚠️ XGBoost Baseline not found")

print(f"\n✓ Total models loaded: {len(models)}")

if len(models) < 2:
    print("\n❌ ERROR: Need at least 2 models for ensemble!")
    print("   Please check model paths and re-run")
    exit()


LOADING TRAINED MODELS
✓ Loaded: GB Optuna (Kaggle: 0.801)
✓ Loaded: XGBoost Optuna (Kaggle: 0.795)

✓ Total models loaded: 2


In [4]:



# =============================================================================
# 3. GENERATE PREDICTIONS FROM EACH MODEL
# =============================================================================

print("\n" + "="*80)
print("GENERATING PREDICTIONS")
print("="*80)

train_predictions = {}
test_predictions = {}

for name, model in models.items():
    print(f"\n--- {name} ---")
    
    # Train predictions (for validation)
    train_pred = model.predict_proba(X)[:, 1]
    train_auc = roc_auc_score(y, train_pred)
    train_predictions[name] = train_pred
    
    # Test predictions
    test_pred = model.predict_proba(X_test)[:, 1]
    test_predictions[name] = test_pred
    
    print(f"  Train AUC: {train_auc:.4f}")
    print(f"  Test predictions - Min: {test_pred.min():.4f}, Max: {test_pred.max():.4f}, Mean: {test_pred.mean():.4f}")

# =============================================================================
# 4. ENSEMBLE STRATEGY 1: SIMPLE AVERAGE
# =============================================================================

print("\n" + "="*80)
print("ENSEMBLE STRATEGY 1: SIMPLE AVERAGE")
print("="*80)

# Average all models equally
train_ensemble_avg = np.mean(list(train_predictions.values()), axis=0)
test_ensemble_avg = np.mean(list(test_predictions.values()), axis=0)

# Validation
ensemble_avg_auc = roc_auc_score(y, train_ensemble_avg)

print(f"\nSimple Average Ensemble:")
print(f"  Train AUC: {ensemble_avg_auc:.4f}")
print(f"  Test predictions - Min: {test_ensemble_avg.min():.4f}, Max: {test_ensemble_avg.max():.4f}, Mean: {test_ensemble_avg.mean():.4f}")

# Compare to individual models
print(f"\n  Individual model train AUCs:")
for name, pred in train_predictions.items():
    auc = roc_auc_score(y, pred)
    print(f"    {name}: {auc:.4f}")

improvement = ensemble_avg_auc - max([roc_auc_score(y, p) for p in train_predictions.values()])
print(f"\n  Ensemble improvement: {improvement:+.4f}")

# =============================================================================
# 5. ENSEMBLE STRATEGY 2: WEIGHTED AVERAGE
# =============================================================================

print("\n" + "="*80)
print("ENSEMBLE STRATEGY 2: WEIGHTED AVERAGE")
print("="*80)

# Weight models by their Kaggle performance
# Update these weights based on your actual Kaggle scores
weights = {
    'gb_optuna': 0.801,      # Best model gets highest weight
    'xgb_optuna': 0.795,
    # 'xgb_baseline': 0.789,
}

# Normalize weights to sum to 1
total_weight = sum([weights.get(name, 0) for name in models.keys()])
normalized_weights = {name: weights.get(name, 0) / total_weight for name in models.keys()}

print("\nWeights (based on Kaggle scores):")
for name, weight in normalized_weights.items():
    print(f"  {name}: {weight:.3f}")

# Weighted average
train_ensemble_weighted = np.zeros(len(X))
test_ensemble_weighted = np.zeros(len(X_test))

for name, weight in normalized_weights.items():
    if name in train_predictions:
        train_ensemble_weighted += weight * train_predictions[name]
        test_ensemble_weighted += weight * test_predictions[name]

# Validation
ensemble_weighted_auc = roc_auc_score(y, train_ensemble_weighted)

print(f"\nWeighted Average Ensemble:")
print(f"  Train AUC: {ensemble_weighted_auc:.4f}")
print(f"  Test predictions - Min: {test_ensemble_weighted.min():.4f}, Max: {test_ensemble_weighted.max():.4f}, Mean: {test_ensemble_weighted.mean():.4f}")

# =============================================================================
# 6. ENSEMBLE STRATEGY 3: RANK AVERAGE
# =============================================================================

print("\n" + "="*80)
print("ENSEMBLE STRATEGY 3: RANK AVERAGE (ROBUST)")
print("="*80)

# Convert predictions to ranks, then average
# This is robust to different prediction scales

from scipy.stats import rankdata

train_ensemble_rank = np.zeros(len(X))
test_ensemble_rank = np.zeros(len(X_test))

for name in models.keys():
    # Rank predictions (higher probability = higher rank)
    train_ranks = rankdata(train_predictions[name]) / len(X)
    test_ranks = rankdata(test_predictions[name]) / len(X_test)
    
    train_ensemble_rank += train_ranks
    test_ensemble_rank += test_ranks

# Average the ranks
train_ensemble_rank /= len(models)
test_ensemble_rank /= len(models)

# Validation
ensemble_rank_auc = roc_auc_score(y, train_ensemble_rank)

print(f"\nRank Average Ensemble:")
print(f"  Train AUC: {ensemble_rank_auc:.4f}")
print(f"  Test predictions - Min: {test_ensemble_rank.min():.4f}, Max: {test_ensemble_rank.max():.4f}, Mean: {test_ensemble_rank.mean():.4f}")

# =============================================================================
# 7. COMPARE ALL STRATEGIES
# =============================================================================

print("\n" + "="*80)
print("ENSEMBLE COMPARISON")
print("="*80)

strategies = {
    'Simple Average': (train_ensemble_avg, test_ensemble_avg),
    'Weighted Average': (train_ensemble_weighted, test_ensemble_weighted),
    'Rank Average': (train_ensemble_rank, test_ensemble_rank)
}

print(f"\n{'Strategy':<20} {'Train AUC':<12} {'Expected Improvement'}")
print("-" * 60)

best_individual_auc = max([roc_auc_score(y, p) for p in train_predictions.values()])

for strategy_name, (train_pred, _) in strategies.items():
    auc = roc_auc_score(y, train_pred)
    improvement = auc - best_individual_auc
    print(f"{strategy_name:<20} {auc:.4f}       {improvement:+.4f}")

# =============================================================================
# 8. SELECT BEST ENSEMBLE & GENERATE SUBMISSION
# =============================================================================

print("\n" + "="*80)
print("CREATING SUBMISSIONS")
print("="*80)

timestamp = datetime.now().strftime("%Y%m%d_%H%M")

# Save all three strategies
for strategy_name, (train_pred, test_pred) in strategies.items():
    
    # Calculate train AUC
    train_auc = roc_auc_score(y, train_pred)
    
    # Create submission
    submission = pd.DataFrame({
        'icustay_id': test_ids,
        'prediction': test_pred
    })
    
    # Save
    strategy_slug = strategy_name.lower().replace(' ', '_')
    filename = f"../../outputs/predictions/ensemble_{strategy_slug}_{timestamp}.csv"
    submission.to_csv(filename, index=False)
    
    print(f"\n✓ Saved: {filename}")
    print(f"  Strategy: {strategy_name}")
    print(f"  Train AUC: {train_auc:.4f}")

# =============================================================================
# 9. RECOMMENDATION
# =============================================================================

print("\n" + "="*80)
print("RECOMMENDATION")
print("="*80)

# Find best strategy
best_strategy = max(strategies.items(), key=lambda x: roc_auc_score(y, x[1][0]))
best_name, (best_train, best_test) = best_strategy
best_auc = roc_auc_score(y, best_train)

print(f"\n🏆 Best Strategy: {best_name}")
print(f"   Train AUC: {best_auc:.4f}")
print(f"   Improvement over best individual: {best_auc - best_individual_auc:+.4f}")

# Expected Kaggle score
# Based on pattern: ensemble usually reduces gap by ~0.01-0.02
expected_kaggle_low = best_auc - 0.09
expected_kaggle_high = best_auc - 0.06

print(f"\n   Expected Kaggle: ~{expected_kaggle_low:.3f} - {expected_kaggle_high:.3f}")

print(f"\n{'='*80}")
print("NEXT STEPS:")
print(f"{'='*80}")

print(f"\n1. Upload all 3 submissions to Kaggle (test each!)")
print(f"2. Most likely winner: Weighted Average or Rank Average")
print(f"3. Expected improvement: 0.801 → 0.81-0.82")
print(f"\n4. If ensemble beats 0.82:")
print(f"   → Try adding LightGBM to the mix")
print(f"   → Try stacking (train meta-model on predictions)")

print("\n" + "="*80)
print("ENSEMBLE COMPLETE!")
print("="*80)

# =============================================================================
# 10. CROSS-VALIDATION CHECK (OPTIONAL BUT RECOMMENDED)
# =============================================================================

print("\n" + "="*80)
print("BONUS: CROSS-VALIDATION CHECK")
print("="*80)

print("\nValidating ensemble with 5-fold CV...")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# We'll validate the weighted ensemble (usually best)
cv_scores = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X, y), 1):
    X_train_fold = X.iloc[train_idx]
    X_val_fold = X.iloc[val_idx]
    y_val_fold = y.iloc[val_idx]
    
    # Generate predictions from each model
    fold_predictions = []
    for name, model in models.items():
        pred = model.predict_proba(X_val_fold)[:, 1]
        weight = normalized_weights[name]
        fold_predictions.append(weight * pred)
    
    # Weighted ensemble
    ensemble_pred = np.sum(fold_predictions, axis=0)
    
    # Score
    fold_auc = roc_auc_score(y_val_fold, ensemble_pred)
    cv_scores.append(fold_auc)
    
    print(f"  Fold {fold}: {fold_auc:.4f}")

cv_mean = np.mean(cv_scores)
cv_std = np.std(cv_scores)

print(f"\n  CV Mean: {cv_mean:.4f} ± {cv_std:.4f}")
print(f"  Expected Kaggle: ~{cv_mean - 0.09:.3f} - {cv_mean - 0.06:.3f}")

if cv_mean > 0.91:
    print("\n  🎉 CV > 0.91 - Strong ensemble! Expect 0.82+")
elif cv_mean > 0.90:
    print("\n  ✅ CV > 0.90 - Good ensemble! Expect 0.81-0.82")
else:
    print("\n  ⚠️ CV < 0.90 - Modest improvement expected")

print("\n" + "="*80)
print("🚀 Good luck! One of these should beat 0.80!")
print("="*80)


GENERATING PREDICTIONS

--- gb_optuna ---
  Train AUC: 0.9790
  Test predictions - Min: 0.0010, Max: 0.9954, Mean: 0.1130

--- xgb_optuna ---
  Train AUC: 0.9728
  Test predictions - Min: 0.0003, Max: 0.9993, Mean: 0.2670

ENSEMBLE STRATEGY 1: SIMPLE AVERAGE

Simple Average Ensemble:
  Train AUC: 0.9801
  Test predictions - Min: 0.0007, Max: 0.9972, Mean: 0.1900

  Individual model train AUCs:
    gb_optuna: 0.9790
    xgb_optuna: 0.9728

  Ensemble improvement: +0.0011

ENSEMBLE STRATEGY 2: WEIGHTED AVERAGE

Weights (based on Kaggle scores):
  gb_optuna: 0.502
  xgb_optuna: 0.498

Weighted Average Ensemble:
  Train AUC: 0.9801
  Test predictions - Min: 0.0007, Max: 0.9972, Mean: 0.1897

ENSEMBLE STRATEGY 3: RANK AVERAGE (ROBUST)

Rank Average Ensemble:
  Train AUC: 0.9774
  Test predictions - Min: 0.0004, Max: 0.9999, Mean: 0.5001

ENSEMBLE COMPARISON

Strategy             Train AUC    Expected Improvement
------------------------------------------------------------
Simple Average   